In [76]:
import pandas as pd
training_df = pd.read_json("/kaggle/input/category-adapter/training_df1.json")
# Drop duplicate URLs and keep the first occurrence
training_df = training_df.drop_duplicates(subset=["url"], keep="first").reset_index(drop=True)

print(f"✅ Training data after removing duplicates: {len(training_df)} rows remain.")

testing_df = pd.read_json("/kaggle/input/category-adapter/testing_df1.json")

print(training_df)

def LoadData(category):
    main_categories_keywords = {
    "ڈکیتی": (
        "ڈکیتی|چوری|راہزنی|لوٹ مار|اسلحہ کے زور پر ڈکیتی|گھروں میں ڈکیتی|"
        "موٹر سائیکل چھیننے|گاڑی چھیننے|پرس چھیننے|چوروں کی واردات|چور گینگ|"
        "بینک ڈکیتی|شاپ لفٹنگ|جیب تراشی|دکان میں چوری|سڑک پر واردات|ڈاکوؤں کا حملہ|لوٹنے کی کوشش"
    ),

    "قتل": (
        "قتل|خون|قتل عام|گولی مار کر قتل|چھری کے وار سے قتل|خنجر مار کر قتل|"
        "گلا دبا کر قتل|سر قلم|قتل کی واردات|اندھا قتل|بے دردی سے قتل|"
        "خونریز واقعہ|قتل کی کوشش|ٹارگٹ کلنگ|غیرت کے نام پر قتل|فائرنگ سے ہلاکت|"
        "زہر دے کر قتل|کلہاڑی سے حملہ|اندھادھند فائرنگ|لاش برآمد"
    ),

    "جنسی زیادتی": (
        "زیادتی|جنسی زیادتی|اجتماعی زیادتی|بداخلاقی|ہراسگی|جنسی ہراسگی|"
        "چھیڑ چھاڑ|بچوں سے زیادتی|خواتین سے زیادتی|ناشائستہ حرکت|"
        "زیادتی کی کوشش|جنسی تشدد|زیادتی کا نشانہ|بچوں کا استحصال|"
        "شادی شدہ خاتون سے زیادتی|ریپ|ریپ کیس|جنسی جرم"
    ),

    "خود کشی": (
        "خودکشی|خودکشی کی کوشش|جان لینے کی کوشش|اپنی جان لے لی|"
        "زہر کھا لیا|تیزاب پی لیا|آگ لگا لی|پھانسی لے لی|"
        "عمارت سے چھلانگ لگا لی|دریا میں چھلانگ لگا دی|ریلوے ٹریک پر کود گیا|"
        "خودسوزی|دماغی دباؤ سے خودکشی|مایوسی میں خودکشی|خودکشی کا واقعہ"
    ),

    "اغوا": (
        "اغوا|اغوا برائے تاوان|بچے کا اغوا|خاتون کا اغوا|"
        "اغوا کی کوشش|نامعلوم افراد کے ہاتھوں اغوا|لاپتہ|غائب|"
        "جبری گمشدگی|بندوق کے زور پر اغوا|اغوا کار|تاوان کی رقم|"
        "اغوا کی واردات|مدرسے کے بچے اغوا|اسکول سے اغوا|اغوا شدہ بچہ|اغوا شدہ خاتون"
    ),

    "دہشت گردی": (
        "دہشت گرد حملہ|دہشت گردی|دھماکہ|خودکش دھماکہ|خودکش حملہ|"
        "بارودی سرنگ دھماکہ|کار بم دھماکہ|موٹر سائیکل بم|دستی بم حملہ|"
        "دہشت گرد کارروائی|دہشت گردوں کا حملہ|شدت پسند حملہ|فائرنگ کا واقعہ|"
        "دہشت گردی کی کوشش|انتہا پسندوں کا حملہ|فورسز پر حملہ|فوجیوں پر حملہ|"
        "طالبان حملہ|داعش حملہ|القاعدہ حملہ|فرقہ وارانہ حملہ|دھماکے سے ہلاکت|دہشت گرد نیٹ ورک|اسلحہ بردار افراد"
    )
}

    keywords = main_categories_keywords[category]
    
    
    # print(training_df.head())
    
    category_data = training_df[training_df['text_label'].apply(lambda x: category in x)]
    # print(category_data.shape)
    non_category_data = training_df[
        training_df['text'].str.contains(keywords, regex=True, na=False) &
        (~training_df.index.isin(category_data.index)) 
    ]

    print("Training Others: ", non_category_data.shape[0])
    print("Training Non Others: ", category_data.shape[0])
    category_df = pd.concat([ category_data, non_category_data])
    # category_df.drop_duplicates(subset=['url'], inplace=True)
    
    
    # test_category_data = testing_df[testing_df['text_label'].apply(lambda x: category in x)]
    
    # test_non_category_data = testing_df[
    #     testing_df['text'].str.contains(keywords, regex=True, na=False) &
    #     (~testing_df.index.isin(category_data.index)) 
    # ]

    # test_category_df = pd.concat([ test_non_category_data, test_category_data, testing_df.sample(200) ])
    # test_category_df = test_category_df.drop_duplicates(subset=["url"], keep="first").reset_index(drop=True)
    # print(category_df.shape)
    # print(test_category_df.shape)

    test_category_df = build_test_category_data(testing_df, category,keywords, 0.3 )
    print("Testing Others: ", test_category_df[test_category_df['text_label'].apply(lambda x: category in x)].shape[0])
    print("Testing Non Others: ", test_category_df[test_category_df['text_label'].apply(lambda x: category not in x)].shape[0])

    return category_df, test_category_df

✅ Training data after removing duplicates: 7760 rows remain.
                                                    url  \
0      https://www.nawaiwaqt.com.pk/29-Nov-2024/1846152   
1     https://dunya.com.pk/index.php/crime/2022-01-1...   
2     https://dunya.com.pk/index.php/crime/2023-03-0...   
3     https://dunya.com.pk/index.php/crime/2016-01-2...   
4      https://www.nawaiwaqt.com.pk/16-Jan-2025/1859634   
...                                                 ...   
7755   https://www.nawaiwaqt.com.pk/29-Aug-2024/1820642   
7756   https://www.nawaiwaqt.com.pk/22-Mar-2024/1774919   
7757  https://dunya.com.pk/index.php/crime/2017-09-2...   
7758   https://www.nawaiwaqt.com.pk/14-Feb-2025/1868403   
7759   https://www.nawaiwaqt.com.pk/03-Apr-2025/1882850   

                                                   text  \
0     موڑکھنڈا (نامہ نگار)چوروں نے زرعی ٹیوب ویلز کی...   
1     فیصل آباد(سٹی رپورٹر)9 سالہ لڑکی کو مبینہ طور...   
2     فیصل آباد(سٹی رپورٹر)تھانہ سرگودھا روڈ کے علاق.

In [77]:
import pandas as pd
import numpy as np

def build_test_category_data(testing_df, category, keywords, noise_ratio=0.15, random_state=42):
    # 1️⃣ Category-specific data
    test_category_data = testing_df[
        testing_df['text_label'].apply(lambda x: category in x)
    ]

    # 2️⃣ Non-category but related data
    test_non_category_data = testing_df[
        testing_df['text'].str.contains(keywords, regex=True, na=False) &
        (~testing_df.index.isin(test_category_data.index))
    ]

    # 3️⃣ Define total size (balanced)
    target_len = min(len(test_category_data), len(test_non_category_data))
    test_category_data = test_category_data.sample(target_len, random_state=random_state)
    test_non_category_data = test_non_category_data.sample(target_len, random_state=random_state)

    # 4️⃣ Combine balanced base dataset
    base_df = pd.concat([test_category_data, test_non_category_data])

    # 5️⃣ Add controlled noise but keep total length same
    noise_size = int(len(base_df) * noise_ratio)
    noise_candidates = testing_df[
        ~testing_df.index.isin(base_df.index)
    ].sample(noise_size, random_state=random_state)

    # Drop same number of rows from non-category before adding noise → keeps total constant
    drop_from_non_cat = test_non_category_data.sample(noise_size, random_state=random_state)
    test_non_category_data = test_non_category_data.drop(drop_from_non_cat.index)

    # Now build final test df with noise injected
    test_category_df = pd.concat([test_category_data, test_non_category_data, noise_candidates])

    # 6️⃣ Deduplicate by URL and shuffle
    test_category_df = (
        test_category_df.drop_duplicates(subset=["url"], keep="first")
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    # 7️⃣ Show counts
    pos = test_category_df['text_label'].apply(lambda x: category in x).sum()
    neg = len(test_category_df) - pos

    print(f"✅ Total Test Samples: {len(test_category_df)}")
    print(f" - Category ({category}): {pos}")
    print(f" - Non-category + noise: {neg}")
    print(f" - Noise ratio: {noise_ratio:.2f}")

    return test_category_df


In [78]:
categories = ["دہشت گردی", "اغوا", "خود کشی", "جنسی زیادتی", 
              "قتل", "ڈکیتی"]

for cat in categories:
    print(f"Category {cat}")
    train_df, test_df = LoadData(cat)
    print(f"Training Data: {train_df.shape[0]}")
    print(f"Testing Data: {test_df.shape[0]}")
    print()
    print()

Category دہشت گردی
Training Others:  485
Training Non Others:  236
✅ Total Test Samples: 118
 - Category (دہشت گردی): 59
 - Non-category + noise: 59
 - Noise ratio: 0.30
Testing Others:  59
Testing Non Others:  59
Training Data: 721
Testing Data: 118


Category اغوا
Training Others:  534
Training Non Others:  402
✅ Total Test Samples: 176
 - Category (اغوا): 88
 - Non-category + noise: 88
 - Noise ratio: 0.30
Testing Others:  88
Testing Non Others:  88
Training Data: 936
Testing Data: 176


Category خود کشی
Training Others:  283
Training Non Others:  143
✅ Total Test Samples: 88
 - Category (خود کشی): 44
 - Non-category + noise: 44
 - Noise ratio: 0.30
Testing Others:  44
Testing Non Others:  44
Training Data: 426
Testing Data: 88


Category جنسی زیادتی
Training Others:  286
Training Non Others:  481
✅ Total Test Samples: 166
 - Category (جنسی زیادتی): 85
 - Non-category + noise: 81
 - Noise ratio: 0.30
Testing Others:  85
Testing Non Others:  81
Training Data: 767
Testing Data: 166




In [28]:
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

def convert_labels(raw_df1, category , mlb=None):
    raw_df = raw_df1.copy()

    main_categories= {
    "ڈکیتی": ["چوری", "ڈکیتی", "ہائی وے ڈکیتی", "راہزنی"],
    "قتل": ["قتل", "خون", "قاتلانہ حملہ"],
    "جنسی زیادتی": ["جنسی زیادتی", "زیادتی", "ہراسانی", "بد فعلی"],
    "خود کشی": ["خود کشی", "جان لینے کی کوشش", "خود کو آگ لگانا"],
    "اغوا": ["اغوا", "بچوں کا اغوا", "زبردستی لے جانا"],
    "دہشت گردی": ["دہشت گردی", "دہشت گرد حملہ", "بم دھماکہ", "خودکش حملہ"]
}



    def normalize_labels(labels):
        normalized = []
        for label in labels:
            label = str(label).strip() if label is not None else ""
            found = False
            if label == "0" or label == 0:
                continue
            for main_cat, variants in main_categories.items():
                if label in variants:
                    if main_cat not in normalized:
                        normalized.append(main_cat)
                    found = True
                    break
            if not found and "other_crime" not in normalized:

                normalized.append("other_crime")
        return sorted(set(normalized))

    raw_df['text_label'] = raw_df['text_label'].apply(normalize_labels)
    raw_df['text_label'] = raw_df['text_label'].apply(
    lambda labels: ["other_crime"] if "other_crime" in labels else labels
)

    all_normalized = [
        "ڈکیتی", "قتل", "جنسی زیادتی", "خود کشی",
         "اغوا", "other_crime", "دہشت گردی",
    ]

    # Fit only if mlb is not passed (for training)

    raw_df['label'] = raw_df['text_label'].apply(lambda x: 1 if category  in x else 0)
    print(category)

    return raw_df



In [29]:
from sklearn.model_selection import train_test_split


In [4]:
!pip install optuna

In [35]:
from transformers import AutoTokenizer

model_name = "urduhack/roberta-urdu-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [36]:
import torch
import numpy as np
from sklearn.metrics import f1_score, classification_report
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import Dataset

# ---------------- Metrics ----------------
def compute_metrics_single_label(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    return {"micro_f1": micro_f1, "macro_f1": macro_f1}

# ---------------- Training Function ----------------
def train_single_label_full_model(train_df, val_df, test_df, tokenizer, model_name,
                                  batch_size=8, max_token=512, is16=False):

    # ---- Dataset Formatting ----
    def tokenize_and_format(batch):
        tokens = tokenizer(
            batch['text'], padding="max_length", truncation=True, max_length=max_token
        )
        tokens["labels"] = batch["label"]
        return tokens

    train_dataset = Dataset.from_pandas(train_df[['text', 'label']], preserve_index=False)
    val_dataset   = Dataset.from_pandas(val_df[['text', 'label']], preserve_index=False)
    test_dataset  = Dataset.from_pandas(test_df[['text', 'label']], preserve_index=False)

    train_dataset = train_dataset.map(tokenize_and_format, batched=True, remove_columns=['text', 'label'])
    val_dataset   = val_dataset.map(tokenize_and_format, batched=True, remove_columns=['text', 'label'])
    test_dataset  = test_dataset.map(tokenize_and_format, batched=True, remove_columns=['text', 'label'])

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    # ---- Load Base Model (FULL training) ----
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=int(np.unique(train_df["label"]).shape[0]),
        problem_type="single_label_classification"
    )

    # ---- Training Arguments ----
    training_args = TrainingArguments(
        output_dir="./results_full_model",
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=5,
        weight_decay=0.01,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=is16,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,           # keep only the latest/best checkpoint
        load_best_model_at_end=True,
        save_only_model=True,         # <<<<<< key fix: do NOT save optimizer/scheduler
        logging_dir="./logs_full_model",
        report_to="none",
        logging_steps=50
    )

    # ---- Trainer with Early Stopping ----
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_single_label,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # ---- Train ----
    trainer.train()

    # ---- Predictions ----
    predictions = trainer.predict(test_dataset)
    logits = predictions.predictions
    labels = np.array(predictions.label_ids)
    preds = np.argmax(logits, axis=1)

    micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)

    print(f"\n[INFO] Micro F1: {micro_f1:.4f}")
    print(f"[INFO] Macro F1: {macro_f1:.4f}")
    print(classification_report(labels, preds, zero_division=0))

    # ---- Save Model & Tokenizer (weights only) ----
    model.save_pretrained("./full_model_single_label")
    tokenizer.save_pretrained("./full_model_single_label")

    return model, tokenizer


In [37]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from sklearn.metrics import accuracy_score, classification_report

def predict_from_hf_model(model, test_df, category, batch_size=16):
    """
    Run predictions on a Hugging Face-hosted sequence classification model.

    Args:
        model_name (str): Hugging Face model name (e.g., "your-username/urdu-classifier").
        test_df (pd.DataFrame): DataFrame with 'text' and 'label' columns.
        category (int or str): Positive label category to evaluate (0 or 1 for binary tasks).
        batch_size (int): Number of samples per batch.

    Returns:
        tuple: (accuracy, classification_report_str)
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model.to(device)
    model.eval()

    max_len = min(tokenizer.model_max_length, 512)

    texts = test_df["text"].astype(str).tolist()
    labels = test_df["text_label"].apply(lambda x: 1 if "other" not in x else 0).astype(int).tolist()

    preds, y_true = [], []

    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            batch_labels = labels[i:i + batch_size]

            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            logits = outputs.logits
            batch_preds = torch.argmax(logits, dim=1).cpu().numpy()

            preds.extend(batch_preds)
            y_true.extend(batch_labels)

    # Metrics
    acc = accuracy_score(y_true, preds)
    report = classification_report(
        y_true, preds,
        target_names=[f"Not {category}", str(category)],
        zero_division=0
    )

    print(f"\n[INFO] Model: {model_name}")
    print(f"[INFO] Accuracy: {acc:.4f}")
    print(f"[INFO] Classification Report:\n{report}")

    return acc, report



In [38]:
categories = ["دہشت گردی", "اغوا", "خود کشی", "جنسی زیادتی", 
              "قتل", "ڈکیتی"]
# categories = [  "event"]

category_mapping = {
    "دہشت گردی": "dehshatgardi",      # terrorism / terrorist attack
    "اغوا": "aghwa",                   # kidnapping / abduction
    "خود کشی": "khudkushi",            # suicide
    "جنسی زیادتی": "jinsi ziyadati",  # sexual assault / rape
    "قتل": "qatl",                     # murder / homicide
    "ڈکیتی": "dakaiti"                 # robbery / theft
}


results = []
for category in categories:
    # train_df, test_df = LoadData(category)
    # print(training_df)
    
    train_df = convert_labels(training_df, category)
    test_df = convert_labels(testing_df, category)
    train_df, val_df = train_test_split(
        train_df,
        test_size=0.1,
        random_state=42,
        shuffle=True
    )
    print(len(training_df))
    def drop_invalid_rows(df, col="text"):
        # Keep only rows where column is a non-empty string
        mask = df[col].apply(lambda x: isinstance(x, str) and x.strip() != "")
        return df[mask].copy()
    
    # Apply to all
    train_df = drop_invalid_rows(train_df, "text")
    val_df   = drop_invalid_rows(val_df, "text")
    test_df  = drop_invalid_rows(test_df, "text")
    model, tokenizer = train_single_label_full_model(train_df, test_df,  val_df, tokenizer, model_name,
            is16=False, batch_size=8)
    results.append(predict_from_hf_model(
        model=model, 
        test_df=test_df,
        category =  category
    ))


    model.push_to_hub(f"the-usan/eng-crime-{category_mapping[category].replace(' ', '_')}-v3")
    tokenizer.push_to_hub(f"the-usan/eng-crime-{category_mapping[category].replace(' ', '_')}-v3")


دہشت گردی
دہشت گردی
7760


Map:   0%|          | 0/6984 [00:00<?, ? examples/s]

Map:   0%|          | 0/1940 [00:00<?, ? examples/s]

Map:   0%|          | 0/776 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/507M [00:00<?, ?B/s]

...

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1
1,0.085900,0.095212,0.976289,0.759225
2,0.070900,0.108113,0.976289,0.743928
3,0.037000,0.129822,0.976289,0.732571
4,0.007100,0.148511,0.976804,0.752123


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



[INFO] Micro F1: 0.9794
[INFO] Macro F1: 0.7090
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       757
           1       0.67      0.32      0.43        19

    accuracy                           0.98       776
   macro avg       0.82      0.66      0.71       776
weighted avg       0.98      0.98      0.98       776


[INFO] Model: urduhack/roberta-urdu-small
[INFO] Accuracy: 0.0201
[INFO] Classification Report:
               precision    recall  f1-score   support

Not دہشت گردی       0.00      0.00      0.00         0
    دہشت گردی       1.00      0.02      0.04      1940

     accuracy                           0.02      1940
    macro avg       0.50      0.01      0.02      1940
 weighted avg       1.00      0.02      0.04      1940



HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-68fd1e41-3ea2f9642c190766443261ca;b99bf674-1d03-48cf-a5f3-e87f57420082)

Invalid username or password.

In [ ]:
model, tokenizer = train_single_label_full_model(train_df, test_df,  val_df, tokenizer, model_name,
            is16=False, batch_size=8)

In [91]:


print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)


Train shape: (8834, 18)
Val shape: (982, 18)
Test shape: (2455, 18)



[INFO] Model: google-bert/bert-base-cased
[INFO] Accuracy: 0.9010
[INFO] Classification Report:
              precision    recall  f1-score   support

   Not other       0.90      0.93      0.92      1431
       other       0.90      0.86      0.88      1024

    accuracy                           0.90      2455
   macro avg       0.90      0.90      0.90      2455
weighted avg       0.90      0.90      0.90      2455



In [39]:
from huggingface_hub import login
login()

Uploading...:   0%|          | 0.00/433M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/the-usan/eng-crime-event-v2/commit/145907b21700fb7c8740a560098217d11904cec6', commit_message='Upload tokenizer', commit_description='', oid='145907b21700fb7c8740a560098217d11904cec6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/the-usan/eng-crime-event-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='the-usan/eng-crime-event-v2'), pr_revision=None, pr_num=None)

In [99]:
import optuna
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from torch.nn.functional import sigmoid

device = "cuda" if torch.cuda.is_available() else "cpu"

# === Urdu Crime Models Map ===
crime_models = {
    # "قتل": "the-usan/urdu-crime-qatal-v2",
    # "ڈکیتی": "the-usan/urdu-crime-dekati-v2",
    # "جنسی زیادتی": "the-usan/urdu-crime-zayadati-v2",
    # "اغوا": "the-usan/urdu-crime-agwa-v2",
    # "دہشت گردی": "the-usan/urdu-crime-dehshatgardi-v2",
    "خود کشی": "the-usan/urdu-crime-khud_khusi-v2"
}


def evaluate_threshold(model, tokenizer, texts, labels, threshold=0.5, batch_size=8):
    """
    Evaluate model predictions at given threshold.
    Handles binary and multi-label outputs safely.
    """
    preds, y_true = [], []
    max_len = min(tokenizer.model_max_length, 512)

    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            batch_labels = labels[i:i + batch_size]

            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            probs = sigmoid(outputs.logits).detach().cpu().numpy()

            # --- Normalize shape ---
            # If model output shape is (B, 2), take positive class probability
            if probs.ndim == 2:
                if probs.shape[1] == 2:
                    probs = probs[:, 1]  # use prob of "positive" class
                elif probs.shape[1] == 1:
                    probs = probs.squeeze(1)
            elif probs.ndim > 2:
                probs = probs.reshape(-1)  # flatten completely

            preds_batch = (probs > threshold).astype(int)

            # Flatten labels if nested lists (e.g. [[0],[1],[0]])
            batch_labels = np.array(batch_labels).reshape(-1).astype(int)

            preds.extend(preds_batch.tolist())
            y_true.extend(batch_labels.tolist())

    preds = np.array(preds).reshape(-1)
    y_true = np.array(y_true).reshape(-1)

    # Safety check
    if len(y_true) != len(preds):
        min_len = min(len(y_true), len(preds))
        y_true, preds = y_true[:min_len], preds[:min_len]

    acc = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds)
    return acc, f1


def objective(trial, model_name, test_df, category):
    """
    Optuna objective: maximize weighted (F1 + Accuracy)
    """
    threshold = trial.suggest_float("threshold", 0.1, 0.9)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()

    texts = test_df["text"].astype(str).tolist()
    labels = test_df["text_label"].apply(lambda x: 1 if category in x else 0).astype(int).tolist()

    acc, f1 = evaluate_threshold(model, tokenizer, texts, labels, threshold)
    combined_score = 0.6 * f1 + 0.4 * acc
    return combined_score


# === Main Loop ===
optimal_thresholds = {}

for category, model_name in crime_models.items():
    print(f"\n🔍 Optimizing threshold for category: {category}")

    _, testing_df1 = LoadData(category)



    # --- Optuna Study ---
    study = optuna.create_study(direction="maximize")
    study.optimize(lambda trial: objective(trial, model_name, testing_df1, category), n_trials=20)

    best_threshold = study.best_params["threshold"]

    # --- Final evaluation ---
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()

    texts = testing_df["text"].astype(str).tolist()
    labels = testing_df["text_label"].apply(lambda x: 1 if category in x else 0).astype(int).tolist()

    acc, f1 = evaluate_threshold(model, tokenizer, texts, labels, threshold=best_threshold)
    combined = 0.6 * f1 + 0.4 * acc

    optimal_thresholds[category] = {
        "model": model_name,
        "best_threshold": best_threshold,
        "accuracy": acc,
        "f1_score": f1,
        "combined_score": combined
    }

    print(f"✅ {category}: threshold = {best_threshold:.3f}, "
          f"Accuracy = {acc:.4f}, F1 = {f1:.4f}, Combined = {combined:.4f}")


# === Final Summary ===
print("\n=== 🏆 Optimal Thresholds Summary ===")
for k, v in optimal_thresholds.items():
    print(f"{k}: threshold = {v['best_threshold']:.3f}, "
          f"Acc = {v['accuracy']:.4f}, F1 = {v['f1_score']:.4f}, "
          f"Combined = {v['combined_score']:.4f}")


[I 2025-10-26 06:03:08,695] A new study created in memory with name: no-name-c8f88a7c-fdf2-482f-b14a-d1f0a78b7e50



🔍 Optimizing threshold for category: خود کشی
Training Others:  283
Training Non Others:  143
✅ Total Test Samples: 88
 - Category (خود کشی): 44
 - Non-category + noise: 44
 - Noise ratio: 0.30
Testing Others:  44
Testing Non Others:  44


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

[I 2025-10-26 06:03:15,751] Trial 0 finished with value: 0.6910658307210031 and parameters: {'threshold': 0.4069831155653121}. Best is trial 0 with value: 0.6910658307210031.
[I 2025-10-26 06:03:17,613] Trial 1 finished with value: 0.6868181818181818 and parameters: {'threshold': 0.5629826277935265}. Best is trial 0 with value: 0.6910658307210031.
[I 2025-10-26 06:03:19,447] Trial 2 finished with value: 0.5909090909090909 and parameters: {'threshold': 0.8755324067566228}. Best is trial 0 with value: 0.6910658307210031.
[I 2025-10-26 06:03:21,424] Trial 3 finished with value: 0.6915742793791575 and parameters: {'threshold': 0.5265093211582211}. Best is trial 3 with value: 0.6915742793791575.
[I 2025-10-26 06:03:23,369] Trial 4 finished with value: 0.6868181818181818 and parameters: {'threshold': 0.5783039284555307}. Best is trial 3 with value: 0.6915742793791575.
[I 2025-10-26 06:03:25,224] Trial 5 finished with value: 0.6836363636363636 and parameters: {'threshold': 0.1694108048109281}

✅ خود کشی: threshold = 0.447, Accuracy = 0.7072, F1 = 0.0955, Combined = 0.3402

=== 🏆 Optimal Thresholds Summary ===
خود کشی: threshold = 0.447, Acc = 0.7072, F1 = 0.0955, Combined = 0.3402


In [ ]:
categories = ["دہشت گردی", "اغوا", "خود کشی", "جنسی زیادتی", 
              "قتل", "ڈکیتی"]

In [14]:
testing_df['new_labels'] = testing_df["text_label"].apply(lambda x: 1 if "جنسی زیادتی" in x else 0).astype(int).tolist()

In [17]:
testing_df['new_labels'].sum()

105

In [16]:
testing_df

,index,url,text,label,text_label,new_labels
0,314,https://www.nawaiwaqt.com.pk/09-May-2025/1893380,پنڈی بھٹیاں (نامہ نگار)کوٹ نکہ روڈ پرٹریفک حاد...,"[0, 0, 0, 0, 0, 1, 0]",[other_crime],0
1,75,https://www.city42.tv/26-Sep-2024/141616,داتا دربار سے اغواء ہونے والی 15سالہ آمنہ سند...,"[0, 0, 0, 0, 0, 1, 0]",[other_crime],0
2,94,https://www.nawaiwaqt.com.pk/03-Sep-2018/897455,لاہور کی سکیورٹی ہائی الرٹ کرنے کا حکم جاری ل...,"[0, 0, 0, 0, 0, 1, 0]",[other_crime],0
3,1853,https://www.nawaiwaqt.com.pk/08-May-2025/1893127,لاہور(خبرنگار)پنجاب بار کونسل کے وائس چیئر مین...,"[0, 0, 0, 0, 0, 0, 1]",[دہشت گردی],0
4,480,https://dailypakistan.com.pk/13-Sep-2018/847192,بیٹی کو قابل اعتراض حالت میں دیکھ کر آشنا سمیت...,"[0, 1, 0, 0, 0, 0, 0]",[قتل],0
...,...,...,...,...,...,...
1935,567,https://www.nawaiwaqt.com.pk/13-Mar-2024/1772209,یوکرائن کے روس میں تیل کی تنصیبات پر ڈرون حم...,"[0, 0, 0, 0, 0, 1, 0]",[other_crime],0
1936,381,https://www.24urdu.com/05-Mar-2021/19141,شوق یاخونی کھیل: قاتل ڈور نے ایک اور جان لے لی...,"[0, 0, 0, 0, 0, 1, 0]",[other_crime],0
1937,706,https://www.nawaiwaqt.com.pk/22-Dec-2024/1852458,نوشہرہ ورکاں :بہنوئی نے سالے کو ٹانگ میں فائر...,"[0, 0, 0, 0, 0, 1, 0]",[other_crime],0
1938,224,https://www.nawaiwaqt.com.pk/15-Mar-2025/1877558,Title: دہشتگردوں کا بنوں میں 2 تھانوں اور ایک...,"[0, 0, 0, 0, 0, 0, 1]",[دہشت گردی],0


In [24]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.nn.functional import sigmoid, softmax

device = "cuda" if torch.cuda.is_available() else "cpu"

def evaluate_model_with_threshold(model_name, test_df, category, threshold=0.5, batch_size=16):
    """
    Evaluate a binary Urdu crime classification model with a custom threshold.
    Works for both 1-logit and 2-logit binary classification heads.
    """
    # Load model + tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()

    texts = test_df["text"].astype(str).tolist()
    labels = test_df["text_label"].apply(lambda x: 1 if category in x else 0).astype(int).tolist()
    preds, y_true = [], []

    max_len = min(tokenizer.model_max_length, 512)

    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=max_len, return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            logits = outputs.logits

            # --- Handle 1-logit or 2-logit model ---
            if logits.shape[1] == 1:
                probs = sigmoid(logits).squeeze().cpu().numpy()
            else:
                probs = softmax(logits, dim=1)[:, 1].cpu().numpy()

            batch_preds = (probs > threshold).astype(int)
            preds.extend(batch_preds)
            y_true.extend(labels[i:i + batch_size])

    preds = np.array(preds)
    y_true = np.array(y_true)

    acc = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds)
    combined = 0.6 * f1 + 0.4 * acc

    report = classification_report(
        y_true, preds,
        target_names=[f"Not {category}", f"{category}"],
        zero_division=0
    )

    print(f"\n📊 Model: {model_name}")
    print(f"🧾 Category: {category}")
    print(f"🔹 Threshold: {threshold:.3f}")
    print(f"✅ Accuracy: {acc:.4f}")
    print(f"✅ F1 Score: {f1:.4f}")
    print(f"⭐ Combined Score: {combined:.4f}")
    print(f"\nClassification Report:\n{report}")

    return {
        "model": model_name,
        "category": category,
        "threshold": threshold,
        "accuracy": acc,
        "f1": f1,
        "combined_score": combined,
        "report": report
    }


In [ ]:
crime_models = {
    "قتل": "the-usan/urdu-crime-qatal-v2",
    "ڈکیتی": "the-usan/urdu-crime-dekati-v2",
    "جنسی زیادتی": "the-usan/urdu-crime-zayadati-v2",
    "اغوا": "the-usan/urdu-crime-agwa-v2",
    "دہشت گردی": "the-usan/urdu-crime-dehshatgardi-v2",
    "خودکشی": "the-usan/urdu-crime-khud_khusi-v2"
}


In [104]:
_, test_df = LoadData("خود کشی")
result = evaluate_model_with_threshold(
    model_name="the-usan/urdu-crime-agwa-v2",
    test_df=test_df,
    category="خود کشی",
    threshold=0.447   
)

print(result["accuracy"], result["f1"])

Training Others:  283
Training Non Others:  143
✅ Total Test Samples: 88
 - Category (خود کشی): 44
 - Non-category + noise: 44
 - Noise ratio: 0.30
Testing Others:  44
Testing Non Others:  44

📊 Model: the-usan/urdu-crime-agwa-v2
🧾 Category: خود کشی
🔹 Threshold: 0.447
✅ Accuracy: 0.5341
✅ F1 Score: 0.3692
⭐ Combined Score: 0.4352

Classification Report:
              precision    recall  f1-score   support

 Not خود کشی       0.52      0.80      0.63        44
     خود کشی       0.57      0.27      0.37        44

    accuracy                           0.53        88
   macro avg       0.55      0.53      0.50        88
weighted avg       0.55      0.53      0.50        88

0.5340909090909091 0.3692307692307692


In [98]:
_, test_df = LoadData("اغوا")
result = evaluate_model_with_threshold(
    model_name="the-usan/urdu-crime-agwa-v2",
    test_df=test_df,
    category="اغوا",
    threshold=0.504   
)

print(result["accuracy"], result["f1"])

Training Others:  534
Training Non Others:  402
✅ Total Test Samples: 176
 - Category (اغوا): 88
 - Non-category + noise: 88
 - Noise ratio: 0.30
Testing Others:  88
Testing Non Others:  88

📊 Model: the-usan/urdu-crime-agwa-v2
🧾 Category: اغوا
🔹 Threshold: 0.504
✅ Accuracy: 0.8295
✅ F1 Score: 0.8256
⭐ Combined Score: 0.8272

Classification Report:
              precision    recall  f1-score   support

    Not اغوا       0.82      0.85      0.83        88
        اغوا       0.85      0.81      0.83        88

    accuracy                           0.83       176
   macro avg       0.83      0.83      0.83       176
weighted avg       0.83      0.83      0.83       176

0.8295454545454546 0.8255813953488372


In [97]:
_, test_df = LoadData("دہشت گردی")
result = evaluate_model_with_threshold(
    model_name="the-usan/urdu-crime-dehshatgardi-v2",
    test_df=test_df,
    category="دہشت گردی",
    threshold=0.486    # your optimized threshold
)

print(result["accuracy"], result["f1"])


Training Others:  485
Training Non Others:  236
✅ Total Test Samples: 118
 - Category (دہشت گردی): 59
 - Non-category + noise: 59
 - Noise ratio: 0.30
Testing Others:  59
Testing Non Others:  59

📊 Model: the-usan/urdu-crime-dehshatgardi-v2
🧾 Category: دہشت گردی
🔹 Threshold: 0.486
✅ Accuracy: 0.8220
✅ F1 Score: 0.7835
⭐ Combined Score: 0.7989

Classification Report:
               precision    recall  f1-score   support

Not دہشت گردی       0.74      1.00      0.85        59
    دہشت گردی       1.00      0.64      0.78        59

     accuracy                           0.82       118
    macro avg       0.87      0.82      0.82       118
 weighted avg       0.87      0.82      0.82       118

0.8220338983050848 0.7835051546391751


In [83]:
_, test_df = LoadData("ڈکیتی")
result = evaluate_model_with_threshold(
    model_name="the-usan/urdu-crime-dekati-v2",
    test_df=test_df,
    category="ڈکیتی",
    threshold=0.858    # your optimized threshold
)

print(result["accuracy"], result["f1"])


Training Others:  581
Training Non Others:  731
✅ Total Test Samples: 306
 - Category (ڈکیتی): 158
 - Non-category + noise: 148
 - Noise ratio: 0.30
Testing Others:  158
Testing Non Others:  148

📊 Model: the-usan/urdu-crime-dekati-v2
🧾 Category: ڈکیتی
🔹 Threshold: 0.858
✅ Accuracy: 0.8268
✅ F1 Score: 0.8307
⭐ Combined Score: 0.8291

Classification Report:
              precision    recall  f1-score   support

   Not ڈکیتی       0.81      0.83      0.82       148
       ڈکیتی       0.84      0.82      0.83       158

    accuracy                           0.83       306
   macro avg       0.83      0.83      0.83       306
weighted avg       0.83      0.83      0.83       306

0.826797385620915 0.8306709265175719


In [86]:
_, test_df = LoadData("جنسی زیادتی")
result = evaluate_model_with_threshold(
    model_name="the-usan/urdu-crime-zayadati-v2",
    test_df=test_df,
    category="جنسی زیادتی",
    threshold=0.60   # your optimized threshold
)

print(result["accuracy"], result["f1"])

Training Others:  286
Training Non Others:  481
✅ Total Test Samples: 166
 - Category (جنسی زیادتی): 85
 - Non-category + noise: 81
 - Noise ratio: 0.30
Testing Others:  85
Testing Non Others:  81

📊 Model: the-usan/urdu-crime-zayadati-v2
🧾 Category: جنسی زیادتی
🔹 Threshold: 0.600
✅ Accuracy: 0.9096
✅ F1 Score: 0.9133
⭐ Combined Score: 0.9118

Classification Report:
                 precision    recall  f1-score   support

Not جنسی زیادتی       0.92      0.89      0.91        81
    جنسی زیادتی       0.90      0.93      0.91        85

       accuracy                           0.91       166
      macro avg       0.91      0.91      0.91       166
   weighted avg       0.91      0.91      0.91       166

0.9096385542168675 0.9132947976878613


In [94]:
_, test_df = LoadData("قتل")
result = evaluate_model_with_threshold(
    model_name="the-usan/urdu-crime-qatal-v2",
    test_df=test_df,
    category="قتل",
    threshold=0.102  # your optimized threshold
)

print(result["accuracy"], result["f1"])

Training Others:  708
Training Non Others:  1182
✅ Total Test Samples: 352
 - Category (قتل): 187
 - Non-category + noise: 165
 - Noise ratio: 0.30
Testing Others:  187
Testing Non Others:  165

📊 Model: the-usan/urdu-crime-qatal-v2
🧾 Category: قتل
🔹 Threshold: 0.102
✅ Accuracy: 0.8551
✅ F1 Score: 0.8618
⭐ Combined Score: 0.8591

Classification Report:
              precision    recall  f1-score   support

     Not قتل       0.84      0.86      0.85       165
         قتل       0.87      0.85      0.86       187

    accuracy                           0.86       352
   macro avg       0.85      0.86      0.85       352
weighted avg       0.86      0.86      0.86       352

0.8551136363636364 0.8617886178861789
